<a href="https://colab.research.google.com/github/franziska1310/aiim_assignment/blob/main/efficientnet_maike.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load the Data

## Set up Paths and Variables


*   Define the paths to the test and trainings data
*   Set Variables such as Batch Size and Image Size



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import f1_score
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm # Import tqdm

from google.colab import drive
drive.mount('/content/drive')

# DATA_DIR = Path('/content/drive/MyDrive/Technical_Assignment/data') # Pfad Franzi
DATA_DIR = Path('/content/drive/MyDrive/AIIM_2026/Technical_Assignment/data') # Pfad Maike
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42
BATCH_SIZE = 32
IMG_SIZE = 224

torch.manual_seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Transform Data
Create transformation for training and testing, respectively

Training:

*   Define Output Channels
*   Horizontal Flip
*   Rotation
*   Color Changes
*   Crop
*   transform to tensor
*   normalize

Validation:
*   Define Output Channels
*   transform to tensor
*   normalize









In [ ]:
train_transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=3),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.RandomAutocontrast(p=0.3), # Added

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## Create Training and Validation Data


*   Determine the intended size of training and validation set
*   do a random split of the training data into training and validation set
*   Transform both subsets according to the defined transformations above





In [ ]:
full_dataset = datasets.ImageFolder(TRAIN_DIR)

val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_idx, val_idx = random_split(
    range(len(full_dataset)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        img = self.transform(img)
        return img, label

train_dataset = TransformSubset(full_dataset, train_idx, train_transform)
val_dataset = TransformSubset(full_dataset, val_idx, val_transform)

## Determine Class Weight

To apply weights to the classes to account for potential imbalances, calculate the class weights




In [ ]:
from collections import Counter
import torch
import numpy as np

train_labels = [full_dataset.targets[i] for i in train_idx]

# for each label, count how often it occurs
class_counts = Counter(train_labels)

# determine the number of classes and the numer of total samples
num_classes = len(full_dataset.classes) # Verwende die Anzahl der Klassen vom full_dataset
total_samples = len(train_labels)

class_weights = []

# for each class, calculate its weight
for i in range(num_classes):
    count = class_counts.get(i, 0)
    if count == 0:
        class_weights.append(total_samples / num_classes)
    else:
        class_weights.append(
            total_samples / (num_classes * count)
        )

class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Class weights:", class_weights)

Class weights: tensor([0.9092, 1.6140, 0.9109, 0.9242, 0.9086], device='cuda:0')


## Setup weighted random sampler

In [ ]:
from collections import Counter
import torch

# 1. Extract labels from your training subset to get accurate counts
train_indices = train_dataset.indices
train_labels = [full_dataset.targets[i] for i in train_indices]

# 2. Count occurrences for each class
class_counts = Counter(train_labels)
# Important: ensure they are in the correct index order (0 to 4)
counts = [class_counts[i] for i in range(len(full_dataset.classes))]

# 3. Calculate the weights
# Formula: total_samples / (num_classes * class_count)
total_samples = len(train_labels)
num_classes = len(full_dataset.classes)
weights = [total_samples / (num_classes * c) for c in counts]

# Convert to tensor and move to your DEVICE (CPU or GPU)
class_weights = torch.tensor(weights, dtype=torch.float).to(DEVICE)

print(f"Calculated Weights: {weights}")
# Expected: Fear (the minority) will have a weight > 1.0, others < 1.0

# 4. Update your Loss Function
criterion = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

Calculated Weights: [0.9091703056768559, 1.613953488372093, 0.910875, 0.9241597970830692, 0.9086034912718205]


In [ ]:
from torch.utils.data import WeightedRandomSampler

# 1. Assign a weight to every individual sample in the training set
# (If an image is 'Fear', it gets the 'Fear' weight)
sample_weights = [weights[label] for label in train_labels]

# 2. Create the sampler
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

## Load training and validation data

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Anzahl Elenemente in den Datasets
print(f"Anzahl Trainingsbeispiele: {len(train_dataset)}")
print(f"Anzahl Validierungsbeispiele: {len(val_dataset)}")

Anzahl Trainingsbeispiele: 7287
Anzahl Validierungsbeispiele: 1821


## Store Data Locally in Session
For faster processing, copy the data to the local session
*   set up paths for local data
*   remove already existing local directories
*   copy the data
*   update paths to use local copies
*   apply prprocessing to local copies of data



In [ ]:
import shutil
import os

LOCAL_DATA_DIR = Path('/content/local_data')
LOCAL_TRAIN_DIR = LOCAL_DATA_DIR / 'train'
LOCAL_TEST_DIR = LOCAL_DATA_DIR / 'test'

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Remove local directories if they exist to prevent FileExistsError
if LOCAL_TRAIN_DIR.exists() and LOCAL_TRAIN_DIR.is_dir():
    shutil.rmtree(LOCAL_TRAIN_DIR)
if LOCAL_TEST_DIR.exists() and LOCAL_TEST_DIR.is_dir():
    shutil.rmtree(LOCAL_TEST_DIR)

# copy data
print(f"Copying data from {TRAIN_DIR} to {LOCAL_TRAIN_DIR}")
shutil.copytree(TRAIN_DIR, LOCAL_TRAIN_DIR)
print(f"Copying data from {TEST_DIR} to {LOCAL_TEST_DIR}")
shutil.copytree(TEST_DIR, LOCAL_TEST_DIR)

# Update dataset paths to use local copies
# Load full dataset from local storage WITHOUT transform
full_dataset_local = datasets.ImageFolder(LOCAL_TRAIN_DIR)

# Re-split the dataset using the local copy (indices)
val_size = int(0.2 * len(full_dataset_local))
train_size = len(full_dataset_local) - val_size

train_indices_local, val_indices_local = random_split(
    range(len(full_dataset_local)),
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Use TransformSubset with local full_dataset and appropriate transforms
train_dataset_local = TransformSubset(
    full_dataset_local,
    train_indices_local,
    train_transform
)

val_dataset_local = TransformSubset(
    full_dataset_local,
    val_indices_local,
    val_transform
)

# Set num_workers to 0 to avoid multiprocessing issues that can cause AssertionErrors in Colab
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset_local, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset_local, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Updated train_loader with num_workers={NUM_WORKERS} and local data path.")
print(f"Anzahl Trainingsbeispiele (local): {len(train_dataset_local)}")
print(f"Anzahl Validierungsbeispiele (local): {len(val_dataset_local)}")

Copying data from /content/drive/MyDrive/AIIM_2026/Technical_Assignment/data/train to /content/local_data/train
Copying data from /content/drive/MyDrive/AIIM_2026/Technical_Assignment/data/test to /content/local_data/test
Updated train_loader with num_workers=0 and local data path.
Anzahl Trainingsbeispiele (local): 7287
Anzahl Validierungsbeispiele (local): 1821


# Build the Model

## Model Setup

*   create an EfficientNet Model instance
*   set the classifier
*   move the model to device



In [ ]:
model = models.efficientnet_b0(pretrained=True)

# ❄️ alles einfrieren
for param in model.parameters():
    param.requires_grad = False

# set the classifier
model.classifier = nn.Sequential(
    nn.Linear(model.classifier[1].in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.4), # Increased dropout rate from 0.3 to 0.4
    nn.Linear(256, 5)
)

# move model to device
model = model.to(DEVICE)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## Training Setup


*   freeze all parameters
*   train only the classifier
*   define the lossfunction
*   define the optimizer and learning rate





In [ ]:
import torch.nn as nn
import torch.optim as optim

# Freeze alles
for param in model.parameters():
    param.requires_grad = False

# Nur classifier trainieren
for param in model.classifier.parameters():
    param.requires_grad = True

# define loss function
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# define optimizer
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4) # könnten hier eine andere Learning Rate testen

## Early Stopping Implementierung

Diese Klasse hilft, Overfitting zu verhindern, indem sie das Training stoppt, wenn der Validierungs-Score über eine bestimmte Anzahl von Epochen hinweg nicht verbessert wird. Sie speichert auch das beste Modell.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0, mode='max', path='best_model.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.path = path
        self.counter = 0
        self.best_score = None
        self.early_stop = False

        if mode == 'max':
            self.val_score = -np.inf
        else:
            self.val_score = np.inf # Corrected typo: np.insertnf to np.inf

    def __call__(self, current_score, model):
        if self.mode == 'max':
            if current_score > self.val_score + self.min_delta:
                self.val_score = current_score
                self.counter = 0
                torch.save(model.state_dict(), self.path) # Speichere das beste Modell
                # print(f"  New best score: {self.val_score:.4f}. Model saved.")
            else:
                self.counter += 1
                # print(f"  Validation score did not improve. Counter: {self.counter}/{self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True
        elif self.mode == 'min':
            # Changed comparison for mode='min' to reflect minimization
            if current_score < self.val_score - self.min_delta:
                self.val_score = current_score
                self.counter = 0
                torch.save(model.state_dict(), self.path) # Speichere das beste Modell
                # print(f"  New best score: {self.val_score:.4f}. Model saved.")
            else:
                self.counter += 1
                # print(f"  Validation score did not improve. Counter: {self.counter}/{self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True

## Training Loop

In [ ]:
def train_epoch(model, loader, epoch_num=0): # epoch_num für tqdm desc
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    # Verwenden von tqdm für den Fortschrittsbalken der Batches
    for images, labels in tqdm(loader, desc=f"Training Epoch {epoch_num+1}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    train_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, train_f1

## F1-Score Evaluation

In [ ]:
def evaluate(model, loader, epoch_num=0): # epoch_num für tqdm desc
    model.eval()

    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        # Verwenden von tqdm für den Fortschrittsbalken der Batches
        for images, labels in tqdm(loader, desc=f"Validation Epoch {epoch_num+1}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels) # Loss auch hier berechnen
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    return avg_loss, val_f1

## First round of training, optimizing just the classifier

In [ ]:
'''
EPOCHS = 15

# Verwenden von tqdm für den Fortschrittsbalken der Epochen
for epoch in tqdm(range(EPOCHS), desc="Phase 1 Training"):
    train_loss, train_f1 = train_epoch(model, train_loader, epoch_num=epoch)
    val_loss, val_f1 = evaluate(model, val_loader, epoch_num=epoch)

    print(f"[Phase1] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train F1={train_f1:.4f} | Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")
'''

'\nEPOCHS = 15\n\n# Verwenden von tqdm für den Fortschrittsbalken der Epochen\nfor epoch in tqdm(range(EPOCHS), desc="Phase 1 Training"):\n    train_loss, train_f1 = train_epoch(model, train_loader, epoch_num=epoch)\n    val_loss, val_f1 = evaluate(model, val_loader, epoch_num=epoch)\n\n    print(f"[Phase1] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train F1={train_f1:.4f} | Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")\n'

In [ ]:
EPOCHS = 20

# Initialisiere Early Stopping für Phase 1
early_stopping_phase1 = EarlyStopping(patience=5, mode='min', path='best_model_phase1.pth') # Changed mode to 'min'

# Verwenden von tqdm für den Fortschrittsbalken der Epochen
for epoch in tqdm(range(EPOCHS), desc="Phase 1 Training"):
    train_loss, train_f1 = train_epoch(model, train_loader, epoch_num=epoch)
    val_loss, val_f1 = evaluate(model, val_loader, epoch_num=epoch)

    print(f"[Phase1] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train F1={train_f1:.4f} | Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")

    early_stopping_phase1(val_loss, model) # Changed to val_loss
    if early_stopping_phase1.early_stop:
        print("Early stopping triggered in Phase 1!")
        break

# Lade das beste Modell von Phase 1
model.load_state_dict(torch.load('best_model_phase1.pth'))
print("Best model from Phase 1 loaded.")

Phase 1 Training:   0%|          | 0/20 [00:00<?, ?it/s]

Training Epoch 1:   0%|          | 0/228 [00:00<?, ?it/s]

Validation Epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

[Phase1] Epoch 1: Train Loss=1.5334, Train F1=0.2211 | Val Loss=1.5152, Val F1=0.2874


Training Epoch 2:   0%|          | 0/228 [00:00<?, ?it/s]

...

Validation Epoch 19:   0%|          | 0/57 [00:00<?, ?it/s]

[Phase1] Epoch 19: Train Loss=1.2698, Train F1=0.4800 | Val Loss=1.2846, Val F1=0.5048


Training Epoch 20:   0%|          | 0/228 [00:00<?, ?it/s]

Validation Epoch 20:   0%|          | 0/57 [00:00<?, ?it/s]

[Phase1] Epoch 20: Train Loss=1.2813, Train F1=0.4766 | Val Loss=1.2727, Val F1=0.5118
Best model from Phase 1 loaded.


## Finetuning on classifier and two feature levels

In [ ]:
print("Starting fine-tuning...")

# Unfreeze more layers for fine-tuning
# We'll unfreeze 'features.6', 'features.7' and the 'classifier'
for name, param in model.named_parameters():
    if "features.6" in name or "features.7" in name or "classifier" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

print("Unfrozen layers for fine-tuning: features.6, features.7, and classifier.")

Starting fine-tuning...
Unfrozen layers for fine-tuning: features.6, features.7, and classifier.


In [ ]:
params_to_update = []
# Learning rate for the later unfrozen feature layers (e.g., features.6, features.7)
lr_features_6_7 = 1e-4
# Learning rate for the classifier
lr_classifier = 3e-4 # This was the previous single LR

# Define weight decay for regularization
weight_decay_value = 1e-4 # Added weight decay

for name, param in model.named_parameters():
    if param.requires_grad:
        if "classifier" in name:
            params_to_update.append({'params': param, 'lr': lr_classifier, 'weight_decay': weight_decay_value})
        elif "features.7" in name or "features.6" in name:
            params_to_update.append({'params': param, 'lr': lr_features_6_7, 'weight_decay': weight_decay_value})
        else:
            # This should ideally not be reached if unfreezing logic is correct
            # Fallback to default classifier LR if any other trainable param is found
            params_to_update.append({'params': param, 'lr': lr_classifier, 'weight_decay': weight_decay_value})

optimizer = optim.Adam(
    params_to_update # Use the parameter groups
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

print(f"Optimizer configured with layer-wise learning rates: classifier={lr_classifier}, features.6/7={lr_features_6_7}")
print(f"Weight decay (L2 Regularization) set to {weight_decay_value} for trainable parameters.")

Optimizer configured with layer-wise learning rates: classifier=0.0003, features.6/7=0.0001
Weight decay (L2 Regularization) set to 0.0001 for trainable parameters.


In [ ]:
'''
EPOCHS_FINE = 20

# Verwenden von tqdm für den Fortschrittsbalken der Epochen
for epoch in tqdm(range(EPOCHS_FINE), desc="Fine-tuning Phase"):
    train_loss, train_f1 = train_epoch(model, train_loader, epoch_num=epoch)
    val_loss, val_f1 = evaluate(model, val_loader, epoch_num=epoch)

    scheduler.step(val_f1)

    print(f"[FT] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train F1={train_f1:.4f} | Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")
'''

'\nEPOCHS_FINE = 20\n\n# Verwenden von tqdm für den Fortschrittsbalken der Epochen\nfor epoch in tqdm(range(EPOCHS_FINE), desc="Fine-tuning Phase"):\n    train_loss, train_f1 = train_epoch(model, train_loader, epoch_num=epoch)\n    val_loss, val_f1 = evaluate(model, val_loader, epoch_num=epoch)\n\n    scheduler.step(val_f1)\n\n    print(f"[FT] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train F1={train_f1:.4f} | Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")\n'

In [ ]:
EPOCHS_FINE = 40

# Initialisiere Early Stopping für Fine-Tuning
early_stopping_finetune = EarlyStopping(patience=7, mode='min', path='best_model_finetune.pth') # Changed mode to 'min'

# Verwenden von tqdm für den Fortschrittsbalken der Epochen
for epoch in tqdm(range(EPOCHS_FINE), desc="Fine-tuning Phase"):
    train_loss, train_f1 = train_epoch(model, train_loader, epoch_num=epoch)
    val_loss, val_f1 = evaluate(model, val_loader, epoch_num=epoch)

    scheduler.step(val_f1)

    print(f"[FT] Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train F1={train_f1:.4f} | Val Loss={val_loss:.4f}, Val F1={val_f1:.4f}")

    early_stopping_finetune(val_loss, model) # Changed to val_loss
    if early_stopping_finetune.early_stop:
        print("Early stopping triggered in Fine-tuning Phase!")
        break

# Lade das beste Modell vom Fine-Tuning
model.load_state_dict(torch.load('best_model_finetune.pth'))
print("Best model from Fine-tuning loaded.")

Fine-tuning Phase:   0%|          | 0/40 [00:00<?, ?it/s]

Training Epoch 1:   0%|          | 0/228 [00:00<?, ?it/s]

Validation Epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

[FT] Epoch 1: Train Loss=1.2191, Train F1=0.5265 | Val Loss=1.1527, Val F1=0.5906


Training Epoch 2:   0%|          | 0/228 [00:00<?, ?it/s]

...

Training Epoch 17:   0%|          | 0/228 [00:00<?, ?it/s]

Validation Epoch 17:   0%|          | 0/57 [00:00<?, ?it/s]

[FT] Epoch 17: Train Loss=0.6704, Train F1=0.8589 | Val Loss=0.9134, Val F1=0.7614


Training Epoch 18:   0%|          | 0/228 [00:00<?, ?it/s]

Validation Epoch 18:   0%|          | 0/57 [00:00<?, ?it/s]

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns

class_names = full_dataset.classes

def plot_confusion_matrix(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images.to(DEVICE))
            preds = torch.argmax(outputs, dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

plot_confusion_matrix(model, val_loader)

# Make Test Predictions

## Create Test Dataset Class
As the testdata has no class labels, we need a separate class to correctly load them

In [ ]:
from PIL import Image
import os
import torch

class TestDataset(torch.utils.data.Dataset):
    def __init__(self, folder, transform):
        """
        folder: Pfad zu Testbildern
        transform: gleiche Transformation wie Validation!
        """
        self.folder = TEST_DIR
        self.transform = transform

        # sortiert für stabile Reihenfolge (wichtig für Kaggle!)
        self.image_paths = sorted(os.listdir(folder))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Dateiname holen
        img_name = self.image_paths[idx]
        img_path = os.path.join(self.folder, img_name)

        # Bild laden
        image = Image.open(img_path).convert("RGB")

        # Transformation anwenden
        image = self.transform(image)

        # Wichtig: wir geben auch den Dateinamen zurück!
        return image, img_name

## Test Loader
we use the same transformation as for the validation set. When we load the data, it is very important that `shuffle` is set to `false`

In [ ]:
test_dataset = TestDataset(TEST_DIR, val_transform)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False   # ❗ niemals shufflen bei Test!
)

## Make the predictions on the evaluation dataset

we just make the predictions, we do not do any training or drop-outs

In [ ]:
model.eval()

all_preds = []
all_filenames = []

with torch.no_grad():  # spart Speicher + schneller
    for images, filenames in test_loader:

        # auf GPU schieben
        images = images.to(DEVICE)

        # Vorwärtsdurchlauf
        outputs = model(images)

        # Klasse mit höchster Wahrscheinlichkeit
        preds = torch.argmax(outputs, dim=1)

        # zurück auf CPU + numpy
        preds = preds.cpu().numpy()

        # sammeln
        all_preds.extend(preds)
        all_filenames.extend(filenames)

## Map the predicted labels to the classnames

In [ ]:
# get the class names
class_names = full_dataset.classes
print("Classes:", class_names)

# map the labels
pred_labels = [class_names[p] for p in all_preds]

## Create CSV-File

In [ ]:
import pandas as pd

submission = pd.DataFrame({
    "Id": all_filenames,
    "emotions": pred_labels
})

# speichern
submission.to_csv("submission.csv", index=False)

print("CSV erfolgreich gespeichert!")
print(submission.head())

# download the file
from google.colab import files
files.download("submission.csv")